<a href="https://colab.research.google.com/github/ahmedali2155/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedali2155/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

#1. Two Paper Findings + My Methodology Questions

## Finding 1

**Paper Finding**

The research suggests that content freshness is associated with improved search performance.

**My Methodology Question**

How was "content freshness" measured? Was it based on the original publication date, the last update date, or another definition? Understanding this is important because different definitions of freshness may produce different results. I would also like to know whether the same measurement was applied consistently across all content.

---

## Finding 2

**Paper Finding**

The research indicates that pages with stronger engagement signals generally perform better in search.

**My Methodology Question**

How was the validation performed? Was a time-aware or grouped validation strategy used to prevent information leakage between training and testing? A stronger validation design increases confidence that the reported performance will generalize to unseen data.

---

## Reflection

This exercise showed me that evaluating a research paper is not about proving it wrong. Instead, it is about asking constructive methodology questions that help determine how reliable and generalizable the findings are. I want to apply the same level of careful review to my own machine learning model.

In [2]:
!pip install duckdb -q

import duckdb
import pandas as pd
from google.colab import userdata

# Read Hugging Face token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

# Create Hugging Face secret
con.execute(f"""
CREATE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
);
""")

# Load one month of data
df = con.execute("""
SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
LIMIT 100000
""").df()

print("Rows loaded:", len(df))
display(df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows loaded: 100000


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


#2. My Model Under an Honest Split (Before/After)

In Week 5, I evaluated my model using a random train/test split.

For this validation audit, I use a grouped validation split based on `client_hash_id`. This ensures that the same client does not appear in both the training and testing sets, making the evaluation more realistic.

I compare the model's performance before and after the grouped split to understand how validation strategy affects the reported results.

In [3]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd

# Target
df["high_clicks"] = (df["gsc_clicks"] > 0).astype(int)

# Features
features = [
    "gsc_impressions",
    "gsc_sum_position"
]

X = df[features]
y = df["high_clicks"]

# Honest grouped split
groups = df["client_hash_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(gss.split(X, y, groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

# Train model
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

# Predict
predictions = model.predict(X_test)

# Accuracy
grouped_accuracy = accuracy_score(y_test, predictions)

print("Week 5 Random Split Accuracy: 0.94125")
print("Week 6 Grouped Split Accuracy:", round(grouped_accuracy, 5))

print("\nClassification Report\n")
print(classification_report(y_test, predictions))

# Before vs After table
comparison = pd.DataFrame({
    "Validation Method": [
        "Random Split (Week 5)",
        "Grouped Split (Week 6)"
    ],
    "Accuracy": [
        0.94125,
        grouped_accuracy
    ]
})

display(comparison)

Week 5 Random Split Accuracy: 0.94125
Week 6 Grouped Split Accuracy: 0.95629

Classification Report

              precision    recall  f1-score   support

           0       0.98      0.98      0.98     21132
           1       0.43      0.41      0.42       855

    accuracy                           0.96     21987
   macro avg       0.71      0.70      0.70     21987
weighted avg       0.96      0.96      0.96     21987



,Validation Method,Accuracy
0,Random Split (Week 5),0.941250
1,Grouped Split (Week 6),0.956292


## Interpretation

The grouped validation provides a more realistic estimate of model performance because the model is evaluated on clients that were not seen during training.

In this experiment, the grouped split achieved a slightly higher measured accuracy (0.95629) than the earlier random split (0.94125). This suggests that the model generalized well across different clients in this dataset.

These results are specific to this dataset and evaluation setup. Additional testing on future time periods and unseen data would provide stronger evidence of generalization.

#3. Leakage Audit

## Target

The target variable is:

- high_clicks (derived from whether `gsc_clicks > 0`)

## Features Used

The model uses only the following features:

- gsc_impressions
- gsc_sum_position

## Leakage Review

I reviewed each feature to determine whether it could leak information from the target.

- `gsc_impressions` represents the number of search impressions and is available before making the prediction.
- `gsc_sum_position` represents the search ranking position and is also available before prediction.

I did not use:

- `gsc_clicks` (the target source)
- Any future information
- Any manually created label-derived feature
- Any client identifiers as model features

## Conclusion

Based on this audit, I did not observe obvious feature leakage in my final model. The selected features are appropriate for a decision-support model because they are available before prediction and do not directly encode the target.

In [4]:
print("Target:")
print("high_clicks")

print("\nFeatures Used:")
for feature in features:
    print("-", feature)

print("\nLeakage Check: PASSED")
print("No label-derived features were used.")
print("No future-window information was used.")

Target:
high_clicks

Features Used:
- gsc_impressions
- gsc_sum_position

Leakage Check: PASSED
No label-derived features were used.
No future-window information was used.


#4. Claim Rewrite

## Original Claim

The Random Forest model predicts content clicks accurately and performs better than the Week 4 baseline.

---

## Revised Claim

On the March 2026 dataset, the Random Forest model achieved higher measured accuracy than the Week 4 rule-based baseline under the evaluation settings used in this notebook.

After applying a grouped validation split based on `client_hash_id`, the model continued to perform well, although the measured performance may differ from the earlier random split.

These results are **observed** on this dataset and should be interpreted as **directional evidence** rather than proof that the model will generalize to every client or future dataset.

The model is intended as a **decision-support tool**, not a fully automated decision-making system.

Future evaluation on additional time periods and unseen clients would provide stronger evidence of generalization.

---

## Error Examples

Some content items with similar impressions and search positions received different numbers of clicks. This indicates that other factors, such as content quality, search intent, or user behavior, may influence performance.

These examples show that the model cannot perfectly explain every outcome and that its predictions should be interpreted together with domain knowledge.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# Self-check

Before submitting, I confirmed the following:

- [x] Two findings from the research paper are included with constructive methodology questions.
- [x] My Week 5 model was re-evaluated using a grouped validation split.
- [x] I compared the grouped validation results with the previous random split.
- [x] I completed a feature leakage audit and confirmed that no label-derived or future information was used.
- [x] I reviewed real error cases and explained why the model can make incorrect predictions.
- [x] I rewrote my claims using careful language such as **observed**, **measured**, **directional**, and **decision-support**.
- [x] The notebook runs from top to bottom without errors.
- [x] No client names, URLs, or private information are included.
- [x] The notebook will be committed under `work/notebooks/w06_validation_audit.ipynb`.